# Volve Oil Production Data Science Project

## Notebook 05: Application Integration and Final System Validation

### Purpose

This notebook prepares the completed analytical workflow for the Streamlit application.

It brings together the modelling-ready producer-well data, final holdout forecasts, well-monitoring outputs, project metadata, and the saved Ridge Regression pipeline. The focus here is integration and reproducibility rather than further model development.

The notebook checks that the required artifacts are available, prepares the datasets used by the application, verifies that the saved model can be loaded and reproduce its original predictions, and records the methodology and limitations that need to be communicated in the interface.

The modelling and final-holdout conclusions established in Notebook 03 are not changed here.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd


PROJECT_ROOT = Path(
    r"F:\DataScience_Projects\Volve_Oil_Production"
)


PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "processed"
)


MODEL_DIR = (
    PROJECT_ROOT
    /
    "models"
)


MODEL_OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "outputs"
    /
    "03_model_development"
)


OPERATIONAL_OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "outputs"
    /
    "04_operational_insights"
)


NOTEBOOK05_OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "outputs"
    /
    "05_application_integration"
)


NOTEBOOK05_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Project root:",
    PROJECT_ROOT
)


print(
    "Notebook 05 output directory:",
    NOTEBOOK05_OUTPUT_DIR
)

Project root: F:\DataScience_Projects\Volve_Oil_Production
Notebook 05 output directory: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration


In [2]:
MODEL_READY_PATH = (
    PROCESSED_DATA_DIR
    /
    "02_modeling_ready_monthly_forecast.csv"
)


FINAL_HOLDOUT_PREDICTIONS_PATH = (
    MODEL_OUTPUT_DIR
    /
    "final_holdout_predictions.csv"
)


FINAL_HOLDOUT_WELL_METRICS_PATH = (
    MODEL_OUTPUT_DIR
    /
    "final_holdout_well_metrics.csv"
)


FINAL_HOLDOUT_POOLED_METRICS_PATH = (
    MODEL_OUTPUT_DIR
    /
    "final_holdout_pooled_metrics.csv"
)


GENERALIZATION_PATH = (
    MODEL_OUTPUT_DIR
    /
    "ridge_vs_persistence_generalization.csv"
)


FINAL_MODEL_PATH = (
    MODEL_DIR
    /
    "ridge_next_month_oil_forecast_pipeline.joblib"
)


DASHBOARD_WELL_SUMMARY_PATH = (
    OPERATIONAL_OUTPUT_DIR
    /
    "dashboard_well_summary.csv"
)


PRODUCTION_MONITORING_PATH = (
    OPERATIONAL_OUTPUT_DIR
    /
    "dashboard_production_monitoring.csv"
)


FORECAST_RELIABILITY_PATH = (
    OPERATIONAL_OUTPUT_DIR
    /
    "dashboard_forecast_reliability.csv"
)


input_artifacts = {
    "Model-ready monthly forecast dataset":
        MODEL_READY_PATH,

    "Final holdout predictions":
        FINAL_HOLDOUT_PREDICTIONS_PATH,

    "Final holdout well metrics":
        FINAL_HOLDOUT_WELL_METRICS_PATH,

    "Final holdout pooled metrics":
        FINAL_HOLDOUT_POOLED_METRICS_PATH,

    "Ridge versus persistence generalization":
        GENERALIZATION_PATH,

    "Frozen Ridge Regression pipeline":
        FINAL_MODEL_PATH,

    "Dashboard well summary":
        DASHBOARD_WELL_SUMMARY_PATH,

    "Production monitoring summary":
        PRODUCTION_MONITORING_PATH,

    "Forecast reliability summary":
        FORECAST_RELIABILITY_PATH
}


input_artifact_check = pd.DataFrame(
    [
        {
            "artifact": name,
            "exists": path.exists(),
            "path": str(path)
        }
        for name, path
        in input_artifacts.items()
    ]
)


display(
    input_artifact_check
)


assert (
    input_artifact_check[
        "exists"
    ]
    .all()
)


print(
    "\nVerified frozen input artifacts:",
    int(
        input_artifact_check[
            "exists"
        ]
        .sum()
    ),
    "/",
    len(
        input_artifact_check
    )
)


print(
    "PASS: all required Notebook 05 input artifacts are available."
)

,artifact,exists,path
0,Model-ready monthly forecast dataset,True,F:\DataScience_Projects\Volve_Oil_Production\d...
1,Final holdout predictions,True,F:\DataScience_Projects\Volve_Oil_Production\o...
2,Final holdout well metrics,True,F:\DataScience_Projects\Volve_Oil_Production\o...
3,Final holdout pooled metrics,True,F:\DataScience_Projects\Volve_Oil_Production\o...
4,Ridge versus persistence generalization,True,F:\DataScience_Projects\Volve_Oil_Production\o...
5,Frozen Ridge Regression pipeline,True,F:\DataScience_Projects\Volve_Oil_Production\m...
6,Dashboard well summary,True,F:\DataScience_Projects\Volve_Oil_Production\o...
7,Production monitoring summary,True,F:\DataScience_Projects\Volve_Oil_Production\o...
8,Forecast reliability summary,True,F:\DataScience_Projects\Volve_Oil_Production\o...



Verified frozen input artifacts: 9 / 9
PASS: all required Notebook 05 input artifacts are available.


## 1. Load and Validate Frozen Analytical Inputs

The final analytical datasets and evaluation outputs generated in the preceding notebooks are loaded without modification.

The purpose of this section is to confirm that the application-integration stage is using the same modelling-ready observations, final-holdout forecasts, well-level metrics, and operational monitoring summaries that were previously validated.

No additional feature engineering, model fitting, prediction generation, or performance evaluation is performed in this section.

In [3]:
model_ready_df = pd.read_csv(
    MODEL_READY_PATH,
    parse_dates=[
        "DATE"
    ]
)


final_holdout_predictions_df = pd.read_csv(
    FINAL_HOLDOUT_PREDICTIONS_PATH,
    parse_dates=[
        "DATE"
    ]
)


final_holdout_well_metrics_df = pd.read_csv(
    FINAL_HOLDOUT_WELL_METRICS_PATH
)


final_holdout_pooled_metrics_df = pd.read_csv(
    FINAL_HOLDOUT_POOLED_METRICS_PATH
)


generalization_df = pd.read_csv(
    GENERALIZATION_PATH
)


dashboard_well_summary_df = pd.read_csv(
    DASHBOARD_WELL_SUMMARY_PATH,
    parse_dates=[
        "DATE"
    ]
)


production_monitoring_df = pd.read_csv(
    PRODUCTION_MONITORING_PATH,
    parse_dates=[
        "DATE"
    ]
)


forecast_reliability_df = pd.read_csv(
    FORECAST_RELIABILITY_PATH
)


loaded_dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "Model-ready forecast data",
            "rows": len(model_ready_df),
            "columns": len(model_ready_df.columns)
        },
        {
            "dataset": "Final holdout predictions",
            "rows": len(final_holdout_predictions_df),
            "columns": len(final_holdout_predictions_df.columns)
        },
        {
            "dataset": "Final holdout well metrics",
            "rows": len(final_holdout_well_metrics_df),
            "columns": len(final_holdout_well_metrics_df.columns)
        },
        {
            "dataset": "Final holdout pooled metrics",
            "rows": len(final_holdout_pooled_metrics_df),
            "columns": len(final_holdout_pooled_metrics_df.columns)
        },
        {
            "dataset": "Generalization comparison",
            "rows": len(generalization_df),
            "columns": len(generalization_df.columns)
        },
        {
            "dataset": "Dashboard well summary",
            "rows": len(dashboard_well_summary_df),
            "columns": len(dashboard_well_summary_df.columns)
        },
        {
            "dataset": "Production monitoring summary",
            "rows": len(production_monitoring_df),
            "columns": len(production_monitoring_df.columns)
        },
        {
            "dataset": "Forecast reliability summary",
            "rows": len(forecast_reliability_df),
            "columns": len(forecast_reliability_df.columns)
        }
    ]
)


display(
    loaded_dataset_summary
)

,dataset,rows,columns
0,Model-ready forecast data,300,29
1,Final holdout predictions,80,6
2,Final holdout well metrics,10,8
3,Final holdout pooled metrics,2,6
4,Generalization comparison,2,5
5,Dashboard well summary,5,20
6,Production monitoring summary,5,11
7,Forecast reliability summary,5,6


### 1.1 Structural Consistency Check

The loaded datasets are checked against the frozen analytical structure established in the preceding notebooks.

The modelling-ready dataset should contain 300 valid next-month forecasting observations across six producer wells, while the final untouched holdout should contain 80 observations across the five core validation wells.

The dashboard datasets should each contain one summary row for every core producer well.

In [4]:
EXPECTED_CORE_WELLS = {
    "15/9-F-1 C",
    "15/9-F-11",
    "15/9-F-12",
    "15/9-F-14",
    "15/9-F-15 D"
}


model_ready_wells = set(
    model_ready_df[
        "NPD_WELL_BORE_NAME"
    ]
    .unique()
)


holdout_wells = set(
    final_holdout_predictions_df[
        "NPD_WELL_BORE_NAME"
    ]
    .unique()
)


assert len(
    model_ready_df
) == 300


assert len(
    model_ready_wells
) == 6


assert len(
    final_holdout_predictions_df
) == 80


assert (
    holdout_wells
    ==
    EXPECTED_CORE_WELLS
)


assert len(
    final_holdout_well_metrics_df
) == 10


assert len(
    final_holdout_pooled_metrics_df
) == 2


assert len(
    generalization_df
) == 2


assert len(
    dashboard_well_summary_df
) == 5


assert len(
    production_monitoring_df
) == 5


assert len(
    forecast_reliability_df
) == 5


assert (
    set(
        dashboard_well_summary_df[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    set(
        production_monitoring_df[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    set(
        forecast_reliability_df[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


holdout_start = (
    final_holdout_predictions_df[
        "DATE"
    ]
    .min()
)


holdout_end = (
    final_holdout_predictions_df[
        "DATE"
    ]
    .max()
)


print(
    "Model-ready observations:",
    len(
        model_ready_df
    )
)


print(
    "Model-ready producer wells:",
    len(
        model_ready_wells
    )
)


print(
    "Final holdout observations:",
    len(
        final_holdout_predictions_df
    )
)


print(
    "Final holdout core wells:",
    len(
        holdout_wells
    )
)


print(
    "Final holdout period:",
    holdout_start.date(),
    "to",
    holdout_end.date()
)


print(
    "\nPASS: frozen analytical dataset structure is consistent."
)

Model-ready observations: 300
Model-ready producer wells: 6
Final holdout observations: 80
Final holdout core wells: 5
Final holdout period: 2015-04-01 to 2016-08-01

PASS: frozen analytical dataset structure is consistent.


### 1.2 Final-Holdout Prediction Integrity

The frozen final-holdout prediction file is checked for the required forecasting fields and missing values.

The application layer will use these predictions for historical forecast comparison only. The holdout observations are not reused for model selection or further tuning.

In [5]:
required_holdout_columns = [
    "NPD_WELL_BORE_NAME",
    "DATE",
    "oil_volume",
    "target_next_month_oil_volume",
    "ridge_prediction",
    "persistence_prediction"
]


missing_holdout_columns = [
    column
    for column
    in required_holdout_columns
    if column
    not in final_holdout_predictions_df.columns
]


assert (
    len(
        missing_holdout_columns
    )
    ==
    0
)


holdout_missing_values = (
    final_holdout_predictions_df[
        required_holdout_columns
    ]
    .isna()
    .sum()
)


display(
    holdout_missing_values
    .rename(
        "missing_values"
    )
    .to_frame()
)


assert (
    holdout_missing_values
    .sum()
    ==
    0
)


assert (
    len(
        final_holdout_predictions_df
    )
    ==
    80
)


print(
    "Required prediction columns:",
    len(
        required_holdout_columns
    )
)


print(
    "Missing required columns:",
    len(
        missing_holdout_columns
    )
)


print(
    "Missing values in required fields:",
    int(
        holdout_missing_values.sum()
    )
)


print(
    "\nPASS: final-holdout prediction data are intact."
)

,missing_values
NPD_WELL_BORE_NAME,0
DATE,0
oil_volume,0
target_next_month_oil_volume,0
ridge_prediction,0
persistence_prediction,0


Required prediction columns: 6
Missing required columns: 0
Missing values in required fields: 0

PASS: final-holdout prediction data are intact.


## 2. Application Historical Producer-Well Dataset

### 2.1 Build the Core Historical Dataset

A historical producer-well dataset is prepared for the application layer using the frozen modelling-ready observations from Notebook 02.

The dataset is restricted to the five core producer wells used in the final validation workflow. It preserves production volume, production age, operating time, production intensity, fluid-ratio indicators, recent production history, and the observed next-month oil volume.

These records represent valid monthly forecasting-origin observations rather than the complete raw Volve production history. No missing ratio values are imputed during application preparation because some unavailable values are structurally associated with zero-production or zero-on-stream conditions.

In [6]:
historical_application_columns = [
    "NPD_WELL_BORE_NAME",
    "DATE",
    "production_age_months",
    "oil_volume",
    "on_stream_hours",
    "oil_per_on_stream_hour",
    "water_cut_pct",
    "gas_oil_ratio",
    "oil_volume_lag_1",
    "oil_volume_lag_2",
    "oil_volume_lag_3",
    "oil_volume_recent_mean_3",
    "target_next_month_oil_volume"
]


missing_history_columns = [
    column
    for column
    in historical_application_columns
    if column not in model_ready_df.columns
]


assert len(
    missing_history_columns
) == 0


application_history_df = (
    model_ready_df[
        model_ready_df[
            "NPD_WELL_BORE_NAME"
        ]
        .isin(
            EXPECTED_CORE_WELLS
        )
    ][
        historical_application_columns
    ]
    .copy()
    .sort_values(
        [
            "NPD_WELL_BORE_NAME",
            "DATE"
        ]
    )
    .reset_index(
        drop=True
    )
)


application_history_df[
    "is_latest_available_origin"
] = (
    application_history_df[
        "DATE"
    ]
    ==
    application_history_df.groupby(
        "NPD_WELL_BORE_NAME"
    )[
        "DATE"
    ]
    .transform(
        "max"
    )
)


print(
    "Application historical rows:",
    len(
        application_history_df
    )
)


print(
    "Core producer wells:",
    application_history_df[
        "NPD_WELL_BORE_NAME"
    ]
    .nunique()
)


display(
    application_history_df.head()
)

Application historical rows: 295
Core producer wells: 5


,NPD_WELL_BORE_NAME,DATE,production_age_months,oil_volume,on_stream_hours,oil_per_on_stream_hour,water_cut_pct,gas_oil_ratio,oil_volume_lag_1,oil_volume_lag_2,oil_volume_lag_3,oil_volume_recent_mean_3,target_next_month_oil_volume,is_latest_available_origin
0,15/9-F-1 C,2014-04-01,0,11142.47,227.50000,48.977890,0.000000,143.409554,NaN,NaN,NaN,NaN,24901.95,False
1,15/9-F-1 C,2014-05-01,1,24901.95,733.83334,33.934067,3.050290,140.399834,11142.47,NaN,NaN,NaN,19617.76,False
2,15/9-F-1 C,2014-06-01,2,19617.76,705.91666,27.790476,9.538214,147.145326,24901.95,11142.47,NaN,18554.060000,15085.68,False
3,15/9-F-1 C,2014-07-01,3,15085.68,742.41666,20.319695,29.273697,149.106023,19617.76,24901.95,11142.47,19868.463333,6970.43,False
4,15/9-F-1 C,2014-08-01,4,6970.43,432.99166,16.098301,39.388514,150.376777,15085.68,19617.76,24901.95,13891.290000,9168.43,False


### 2.2 Historical Dataset Integrity and Coverage

The application history is checked for well-date uniqueness, expected observation counts, chronological coverage, and missing-value structure.

The five wells have different production histories and therefore different numbers of valid forecasting origins. These differences are preserved rather than forcing a common observation window.

Missing fluid-ratio and production-intensity values are reported but not automatically replaced because their absence may reflect production-state conditions identified during the earlier structural analysis.

In [7]:
EXPECTED_CORE_OBSERVATION_COUNTS = {
    "15/9-F-1 C": 24,
    "15/9-F-11": 38,
    "15/9-F-12": 103,
    "15/9-F-14": 98,
    "15/9-F-15 D": 32
}


duplicate_well_dates = (
    application_history_df
    .duplicated(
        subset=[
            "NPD_WELL_BORE_NAME",
            "DATE"
        ]
    )
    .sum()
)


well_history_summary = (
    application_history_df
    .groupby(
        "NPD_WELL_BORE_NAME"
    )
    .agg(
        observations=(
            "DATE",
            "size"
        ),
        first_origin=(
            "DATE",
            "min"
        ),
        latest_origin=(
            "DATE",
            "max"
        ),
        latest_production_age_months=(
            "production_age_months",
            "max"
        )
    )
    .reset_index()
)


actual_core_observation_counts = (
    application_history_df[
        "NPD_WELL_BORE_NAME"
    ]
    .value_counts()
    .to_dict()
)


assert len(
    application_history_df
) == 295


assert (
    set(
        application_history_df[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    actual_core_observation_counts
    ==
    EXPECTED_CORE_OBSERVATION_COUNTS
)


assert duplicate_well_dates == 0


assert (
    application_history_df[
        "target_next_month_oil_volume"
    ]
    .notna()
    .all()
)


display(
    well_history_summary
)


print(
    "Duplicate well-date observations:",
    int(
        duplicate_well_dates
    )
)


print(
    "\nMissing values retained for application context:"
)


display(
    application_history_df[
        [
            "oil_volume",
            "on_stream_hours",
            "oil_per_on_stream_hour",
            "water_cut_pct",
            "gas_oil_ratio",
            "target_next_month_oil_volume"
        ]
    ]
    .isna()
    .sum()
    .rename(
        "missing_values"
    )
    .to_frame()
)


print(
    "\nPASS: application historical dataset structure is consistent."
)

,NPD_WELL_BORE_NAME,observations,first_origin,latest_origin,latest_production_age_months
0,15/9-F-1 C,24,2014-04-01,2016-03-01,23
1,15/9-F-11,38,2013-07-01,2016-08-01,37
2,15/9-F-12,103,2008-02-01,2016-08-01,102
3,15/9-F-14,98,2008-07-01,2016-08-01,97
4,15/9-F-15 D,32,2014-01-01,2016-08-01,31


Duplicate well-date observations: 0

Missing values retained for application context:


,missing_values
oil_volume,0
on_stream_hours,0
oil_per_on_stream_hour,4
water_cut_pct,4
gas_oil_ratio,4
target_next_month_oil_volume,0



PASS: application historical dataset structure is consistent.


### 2.3 Export of Application Historical Data

The validated producer-well history is exported as an application-ready CSV file.

This dataset will support historical production visualizations and well-level contextual displays in the Streamlit interface. The analytical values remain unchanged from the frozen modelling-ready dataset.

In [8]:
APPLICATION_HISTORY_PATH = (
    NOTEBOOK05_OUTPUT_DIR
    /
    "app_historical_production.csv"
)


application_history_df.to_csv(
    APPLICATION_HISTORY_PATH,
    index=False
)


assert (
    APPLICATION_HISTORY_PATH.exists()
)


exported_history_check = pd.read_csv(
    APPLICATION_HISTORY_PATH,
    parse_dates=[
        "DATE"
    ]
)


assert len(
    exported_history_check
) == 295


assert (
    set(
        exported_history_check[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


print(
    "Saved:",
    APPLICATION_HISTORY_PATH
)


print(
    "Rows:",
    len(
        exported_history_check
    )
)


print(
    "Columns:",
    len(
        exported_history_check.columns
    )
)


print(
    "\nPASS: application historical production dataset exported successfully."
)

Saved: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration\app_historical_production.csv
Rows: 295
Columns: 14

PASS: application historical production dataset exported successfully.


## 3. Application Forecast Comparison Dataset

### 3.1 Prepare Forecast-History Records

The frozen final-holdout predictions from Notebook 03 are prepared for application-level visualization and comparison.

Each record represents a forecasting origin at month \(t\), the observed oil volume in the following month \(t+1\), the naive persistence forecast, and the selected Ridge Regression forecast.

Additional fields are derived only for visualization and interpretation, including the target month, absolute forecast errors, forecast-error difference, and whether the Ridge forecast is physically negative.

No forecasts are regenerated, clipped, corrected, or used for additional model selection.

In [9]:
app_forecast_comparison_df = (
    final_holdout_predictions_df[
        [
            "NPD_WELL_BORE_NAME",
            "DATE",
            "oil_volume",
            "target_next_month_oil_volume",
            "ridge_prediction",
            "persistence_prediction"
        ]
    ]
    .copy()
    .sort_values(
        [
            "NPD_WELL_BORE_NAME",
            "DATE"
        ]
    )
    .reset_index(
        drop=True
    )
)


# Forecast origin DATE represents month t.
# The observed target belongs to the following month.
app_forecast_comparison_df[
    "target_month"
] = (
    app_forecast_comparison_df[
        "DATE"
    ]
    +
    pd.DateOffset(
        months=1
    )
)


app_forecast_comparison_df[
    "ridge_absolute_error"
] = (
    app_forecast_comparison_df[
        "ridge_prediction"
    ]
    -
    app_forecast_comparison_df[
        "target_next_month_oil_volume"
    ]
).abs()


app_forecast_comparison_df[
    "persistence_absolute_error"
] = (
    app_forecast_comparison_df[
        "persistence_prediction"
    ]
    -
    app_forecast_comparison_df[
        "target_next_month_oil_volume"
    ]
).abs()


app_forecast_comparison_df[
    "ridge_minus_persistence_absolute_error"
] = (
    app_forecast_comparison_df[
        "ridge_absolute_error"
    ]
    -
    app_forecast_comparison_df[
        "persistence_absolute_error"
    ]
)


app_forecast_comparison_df[
    "ridge_negative_prediction"
] = (
    app_forecast_comparison_df[
        "ridge_prediction"
    ]
    <
    0
)


print(
    "Application forecast records:",
    len(
        app_forecast_comparison_df
    )
)


print(
    "Producer wells:",
    app_forecast_comparison_df[
        "NPD_WELL_BORE_NAME"
    ]
    .nunique()
)


print(
    "Negative Ridge forecasts:",
    int(
        app_forecast_comparison_df[
            "ridge_negative_prediction"
        ]
        .sum()
    )
)


display(
    app_forecast_comparison_df.head()
)

Application forecast records: 80
Producer wells: 5
Negative Ridge forecasts: 12


,NPD_WELL_BORE_NAME,DATE,oil_volume,target_next_month_oil_volume,ridge_prediction,persistence_prediction,target_month,ridge_absolute_error,persistence_absolute_error,ridge_minus_persistence_absolute_error,ridge_negative_prediction
0,15/9-F-1 C,2015-04-01,3924.18,3832.69,3195.922890,3924.18,2015-05-01,636.767110,91.49,545.277110,False
1,15/9-F-1 C,2015-05-01,3832.69,6344.50,7041.379235,3832.69,2015-06-01,696.879235,2511.81,-1814.930765,False
2,15/9-F-1 C,2015-06-01,6344.50,2093.95,2305.050679,6344.50,2015-07-01,211.100679,4250.55,-4039.449321,False
3,15/9-F-1 C,2015-07-01,2093.95,2871.02,-941.015686,2093.95,2015-08-01,3812.035686,777.07,3034.965686,True
4,15/9-F-1 C,2015-08-01,2871.02,6460.27,5820.796656,2871.02,2015-09-01,639.473344,3589.25,-2949.776656,False


### 3.2 Forecast Comparison Integrity Check

The application forecast dataset is checked against the frozen final-holdout conclusions from Notebook 03.

The recalculated error values are used only to verify that the application dataset reproduces the same 80 observations and forecast outcomes. They do not constitute a new model evaluation or model-selection stage.

In [10]:
ridge_application_mae = (
    app_forecast_comparison_df[
        "ridge_absolute_error"
    ]
    .mean()
)


persistence_application_mae = (
    app_forecast_comparison_df[
        "persistence_absolute_error"
    ]
    .mean()
)


assert len(
    app_forecast_comparison_df
) == 80


assert (
    set(
        app_forecast_comparison_df[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    app_forecast_comparison_df[
        "ridge_negative_prediction"
    ]
    .sum()
    ==
    12
)


assert np.isclose(
    ridge_application_mae,
    3370.2774,
    atol=0.001
)


assert np.isclose(
    persistence_application_mae,
    1786.6786,
    atol=0.001
)


duplicate_forecast_origins = (
    app_forecast_comparison_df
    .duplicated(
        subset=[
            "NPD_WELL_BORE_NAME",
            "DATE"
        ]
    )
    .sum()
)


assert duplicate_forecast_origins == 0


print(
    "Ridge final-holdout MAE:",
    round(
        ridge_application_mae,
        4
    )
)


print(
    "Persistence final-holdout MAE:",
    round(
        persistence_application_mae,
        4
    )
)


print(
    "Negative Ridge forecasts:",
    int(
        app_forecast_comparison_df[
            "ridge_negative_prediction"
        ]
        .sum()
    )
)


print(
    "Duplicate well-origin records:",
    int(
        duplicate_forecast_origins
    )
)


print(
    "\nPASS: application forecast data reproduce the frozen final-holdout results."
)

Ridge final-holdout MAE: 3370.2774
Persistence final-holdout MAE: 1786.6786
Negative Ridge forecasts: 12
Duplicate well-origin records: 0

PASS: application forecast data reproduce the frozen final-holdout results.


### 3.3 Export of Application Forecast Comparison Data

The validated final-holdout forecast records are exported for use in the application layer.

The resulting dataset supports interactive comparison of observed next-month oil production, naive persistence forecasts, Ridge Regression forecasts, and their corresponding errors.

Negative Ridge predictions are preserved exactly as produced during the frozen final evaluation and are identified through a separate diagnostic flag.

In [11]:
APPLICATION_FORECAST_PATH = (
    NOTEBOOK05_OUTPUT_DIR
    /
    "app_forecast_comparison.csv"
)


app_forecast_comparison_df.to_csv(
    APPLICATION_FORECAST_PATH,
    index=False
)


assert (
    APPLICATION_FORECAST_PATH.exists()
)


exported_forecast_check = pd.read_csv(
    APPLICATION_FORECAST_PATH,
    parse_dates=[
        "DATE",
        "target_month"
    ]
)


assert len(
    exported_forecast_check
) == 80


assert (
    exported_forecast_check[
        "ridge_negative_prediction"
    ]
    .sum()
    ==
    12
)


print(
    "Saved:",
    APPLICATION_FORECAST_PATH
)


print(
    "Rows:",
    len(
        exported_forecast_check
    )
)


print(
    "Columns:",
    len(
        exported_forecast_check.columns
    )
)


print(
    "\nPASS: application forecast comparison dataset exported successfully."
)

Saved: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration\app_forecast_comparison.csv
Rows: 80
Columns: 11

PASS: application forecast comparison dataset exported successfully.


## 4. Integrated Well Monitoring and Reliability Dataset

The producer-well monitoring outputs generated in Notebook 04 are consolidated for use in the application layer.

The integrated dataset combines each well's latest available production condition with its frozen final-holdout forecasting reliability indicators.

Production behaviour and forecasting reliability remain separate dimensions. No composite risk score, engineering alarm level, or automated operational recommendation is introduced.

The latest available monitoring origin is preserved independently for each well because the available dates are not identical across all five core producers.

### 4.1 Notebook 04 Component Reconciliation

The production-monitoring and forecast-reliability component tables are checked against the comprehensive dashboard well summary exported from Notebook 04.

This ensures that the application layer does not combine inconsistent versions of the frozen operational outputs.

In [12]:
production_component_columns = [
    "NPD_WELL_BORE_NAME",
    "DATE",
    "oil_volume",
    "oil_volume_change_pct",
    "recent_3_observation_mean_oil",
    "recent_3_zero_observations",
    "on_stream_hours",
    "water_cut_pct",
    "latest_production_state",
    "recent_direction",
    "ratio_measurement_state"
]


reliability_component_columns = [
    "NPD_WELL_BORE_NAME",
    "persistence_WAPE_pct",
    "ridge_WAPE_pct",
    "ridge_minus_persistence_WAPE_points",
    "negative_ridge_forecasts",
    "negative_ridge_forecast_pct"
]


dashboard_production_check = (
    dashboard_well_summary_df[
        production_component_columns
    ]
    .sort_values(
        "NPD_WELL_BORE_NAME"
    )
    .reset_index(
        drop=True
    )
)


production_component_check = (
    production_monitoring_df[
        production_component_columns
    ]
    .sort_values(
        "NPD_WELL_BORE_NAME"
    )
    .reset_index(
        drop=True
    )
)


dashboard_reliability_check = (
    dashboard_well_summary_df[
        reliability_component_columns
    ]
    .sort_values(
        "NPD_WELL_BORE_NAME"
    )
    .reset_index(
        drop=True
    )
)


reliability_component_check = (
    forecast_reliability_df[
        reliability_component_columns
    ]
    .sort_values(
        "NPD_WELL_BORE_NAME"
    )
    .reset_index(
        drop=True
    )
)


pd.testing.assert_frame_equal(
    dashboard_production_check,
    production_component_check,
    check_dtype=False
)


pd.testing.assert_frame_equal(
    dashboard_reliability_check,
    reliability_component_check,
    check_dtype=False
)


print(
    "Production-monitoring rows:",
    len(
        production_component_check
    )
)


print(
    "Forecast-reliability rows:",
    len(
        reliability_component_check
    )
)


print(
    "\nPASS: Notebook 04 dashboard component tables reconcile exactly."
)

Production-monitoring rows: 5
Forecast-reliability rows: 5

PASS: Notebook 04 dashboard component tables reconcile exactly.


### 4.2 Application Well-Monitoring Table

The comprehensive Notebook 04 producer-well summary is converted into an application-ready monitoring table.

Only presentation-oriented fields and direct logical indicators are added. These include a shortened well display name, a formatted monitoring-origin label, zero-production status, ratio-data availability, whether persistence achieved lower final-holdout WAPE than Ridge Regression, and whether any negative Ridge forecasts occurred for the well.

These indicators directly reflect existing analytical results and do not constitute new engineering classifications.

In [13]:
app_well_monitoring_df = (
    dashboard_well_summary_df
    .copy()
    .sort_values(
        "NPD_WELL_BORE_NAME"
    )
    .reset_index(
        drop=True
    )
)


app_well_monitoring_df[
    "display_well_name"
] = (
    app_well_monitoring_df[
        "NPD_WELL_BORE_NAME"
    ]
    .str.replace(
        "15/9-",
        "",
        regex=False
    )
)


app_well_monitoring_df[
    "monitoring_origin_label"
] = (
    app_well_monitoring_df[
        "DATE"
    ]
    .dt.strftime(
        "%Y-%m"
    )
)


app_well_monitoring_df[
    "zero_production_flag"
] = (
    app_well_monitoring_df[
        "oil_volume"
    ]
    <=
    0
)


app_well_monitoring_df[
    "ratio_indicators_available_flag"
] = (
    app_well_monitoring_df[
        "ratio_measurement_state"
    ]
    ==
    "Current ratio indicators available"
)


app_well_monitoring_df[
    "persistence_lower_wape_flag"
] = (
    app_well_monitoring_df[
        "persistence_WAPE_pct"
    ]
    <
    app_well_monitoring_df[
        "ridge_WAPE_pct"
    ]
)


app_well_monitoring_df[
    "ridge_negative_forecast_flag"
] = (
    app_well_monitoring_df[
        "negative_ridge_forecasts"
    ]
    >
    0
)


application_monitoring_view = (
    app_well_monitoring_df[
        [
            "display_well_name",
            "DATE",
            "monitoring_origin_label",
            "oil_volume",
            "oil_volume_change_pct",
            "recent_3_observation_mean_oil",
            "on_stream_hours",
            "water_cut_pct",
            "zero_production_flag",
            "ratio_indicators_available_flag",
            "persistence_WAPE_pct",
            "ridge_WAPE_pct",
            "negative_ridge_forecasts",
            "negative_ridge_forecast_pct",
            "persistence_lower_wape_flag",
            "ridge_negative_forecast_flag"
        ]
    ]
)


display(
    application_monitoring_view.round({
        "oil_volume": 2,
        "oil_volume_change_pct": 2,
        "recent_3_observation_mean_oil": 2,
        "on_stream_hours": 2,
        "water_cut_pct": 2,
        "persistence_WAPE_pct": 2,
        "ridge_WAPE_pct": 2,
        "negative_ridge_forecast_pct": 2
    })
)

,display_well_name,DATE,monitoring_origin_label,oil_volume,oil_volume_change_pct,recent_3_observation_mean_oil,on_stream_hours,water_cut_pct,zero_production_flag,ratio_indicators_available_flag,persistence_WAPE_pct,ridge_WAPE_pct,negative_ridge_forecasts,negative_ridge_forecast_pct,persistence_lower_wape_flag,ridge_negative_forecast_flag
0,F-1 C,2016-03-01,2016-03,2917.50,-38.16,3538.99,356.76,82.13,False,True,44.16,58.56,4,33.33,True,True
1,F-11,2016-08-01,2016-08,14583.73,1.38,14519.74,743.67,85.75,False,True,13.63,18.57,0,0.00,True,False
2,F-12,2016-08-01,2016-08,1442.03,-68.19,3794.72,285.89,88.35,False,True,15.20,15.99,0,0.00,True,False
3,F-14,2016-08-01,2016-08,0.00,-100.00,1467.30,0.00,NaN,True,False,15.43,95.82,1,5.88,True,True
4,F-15 D,2016-08-01,2016-08,0.00,-100.00,1430.83,0.00,NaN,True,False,37.35,102.12,7,41.18,True,True


### 4.3 Export of Application Well-Monitoring Data

The integrated producer-well monitoring dataset is validated and exported for the Streamlit application.

The exported table provides one record for each of the five core producer wells and combines the latest available production context with frozen forecasting-reliability evidence.

In [14]:
assert len(
    app_well_monitoring_df
) == 5


assert (
    set(
        app_well_monitoring_df[
            "NPD_WELL_BORE_NAME"
        ]
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    app_well_monitoring_df[
        "NPD_WELL_BORE_NAME"
    ]
    .duplicated()
    .sum()
    ==
    0
)


assert (
    app_well_monitoring_df[
        "persistence_lower_wape_flag"
    ]
    .all()
)


assert (
    app_well_monitoring_df[
        "negative_ridge_forecasts"
    ]
    .sum()
    ==
    12
)


assert (
    app_well_monitoring_df[
        "zero_production_flag"
    ]
    .sum()
    ==
    2
)


assert (
    app_well_monitoring_df[
        "ratio_indicators_available_flag"
    ]
    .sum()
    ==
    3
)


APPLICATION_MONITORING_PATH = (
    NOTEBOOK05_OUTPUT_DIR
    /
    "app_well_monitoring.csv"
)


app_well_monitoring_df.to_csv(
    APPLICATION_MONITORING_PATH,
    index=False
)


assert (
    APPLICATION_MONITORING_PATH.exists()
)


exported_monitoring_check = pd.read_csv(
    APPLICATION_MONITORING_PATH,
    parse_dates=[
        "DATE"
    ]
)


assert len(
    exported_monitoring_check
) == 5


print(
    "Saved:",
    APPLICATION_MONITORING_PATH
)


print(
    "Rows:",
    len(
        exported_monitoring_check
    )
)


print(
    "Columns:",
    len(
        exported_monitoring_check.columns
    )
)


print(
    "Zero-production latest states:",
    int(
        exported_monitoring_check[
            "zero_production_flag"
        ]
        .sum()
    )
)


print(
    "Wells with negative Ridge forecasts:",
    int(
        exported_monitoring_check[
            "ridge_negative_forecast_flag"
        ]
        .sum()
    )
)


print(
    "\nPASS: application well-monitoring dataset exported successfully."
)

Saved: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration\app_well_monitoring.csv
Rows: 5
Columns: 26
Zero-production latest states: 2
Wells with negative Ridge forecasts: 3

PASS: application well-monitoring dataset exported successfully.


## 5. Saved Model and Pipeline Verification

The Ridge Regression pipeline saved in Notebook 03 is loaded directly from the model directory and checked before it is used by the application.

The aim is to confirm that the saved pipeline, expected feature structure, and preprocessing steps remain usable without retraining the model.

### 5.1 Load the Saved Pipeline

The serialized Joblib pipeline is loaded from the model directory and its pipeline structure is inspected.

In [15]:
from sklearn.pipeline import Pipeline


assert FINAL_MODEL_PATH.exists()


frozen_ridge_pipeline = joblib.load(
    FINAL_MODEL_PATH
)


print(
    "Model path:",
    FINAL_MODEL_PATH
)


print(
    "Loaded object type:",
    type(
        frozen_ridge_pipeline
    ).__name__
)


assert isinstance(
    frozen_ridge_pipeline,
    Pipeline
)


print(
    "Pipeline steps:",
    [
        step_name
        for step_name, _
        in frozen_ridge_pipeline.steps
    ]
)


print(
    "\nPASS: frozen forecasting pipeline loaded successfully."
)

Model path: F:\DataScience_Projects\Volve_Oil_Production\models\ridge_next_month_oil_forecast_pipeline.joblib
Loaded object type: Pipeline
Pipeline steps: ['preprocessor', 'model']

PASS: frozen forecasting pipeline loaded successfully.


### 5.2 Model and Feature-Schema Check

The loaded pipeline is checked against the feature set used for the final Ridge model.

The model uses ten numerical production and history variables together with producer-well identity. Neither the target value nor future operating information is included as an input.

In [16]:
CORE_NUMERIC_FEATURES = [
    "oil_volume",
    "production_age_months",
    "oil_per_on_stream_hour",
    "water_cut_pct",
    "gas_oil_ratio",
    "on_stream_hours",
    "oil_volume_lag_1",
    "oil_volume_lag_2",
    "oil_volume_lag_3",
    "oil_volume_recent_mean_3"
]


CATEGORICAL_FEATURES = [
    "NPD_WELL_BORE_NAME"
]


APPLICATION_MODEL_FEATURES = (
    CATEGORICAL_FEATURES
    +
    CORE_NUMERIC_FEATURES
)


missing_model_features = [
    feature
    for feature
    in APPLICATION_MODEL_FEATURES
    if feature not in model_ready_df.columns
]


assert len(
    missing_model_features
) == 0


final_estimator_name = (
    frozen_ridge_pipeline
    .steps[-1][1]
    .__class__.__name__
)


print(
    "Final estimator:",
    final_estimator_name
)


print(
    "Numerical features:",
    len(
        CORE_NUMERIC_FEATURES
    )
)


print(
    "Categorical features:",
    len(
        CATEGORICAL_FEATURES
    )
)


print(
    "Total application input features:",
    len(
        APPLICATION_MODEL_FEATURES
    )
)


print(
    "\nApplication model features:"
)


for feature in APPLICATION_MODEL_FEATURES:
    print(
        "-",
        feature
    )


assert (
    final_estimator_name
    ==
    "Ridge"
)


final_estimator = (
    frozen_ridge_pipeline
    .steps[-1][1]
)


if hasattr(
    final_estimator,
    "alpha"
):
    print(
        "\nFrozen Ridge alpha:",
        final_estimator.alpha
    )

    assert np.isclose(
        float(
            final_estimator.alpha
        ),
        10.0
    )


print(
    "\nPASS: frozen model and application feature schema are consistent."
)

Final estimator: Ridge
Numerical features: 10
Categorical features: 1
Total application input features: 11

Application model features:
- NPD_WELL_BORE_NAME
- oil_volume
- production_age_months
- oil_per_on_stream_hour
- water_cut_pct
- gas_oil_ratio
- on_stream_hours
- oil_volume_lag_1
- oil_volume_lag_2
- oil_volume_lag_3
- oil_volume_recent_mean_3

Frozen Ridge alpha: 10.0

PASS: frozen model and application feature schema are consistent.


### 5.3 Reconstruct Final-Holdout Inputs

The 80 final-holdout forecasting origins are matched to their predictor values in the modelling-ready dataset.

This gives a known set of inputs for checking whether the saved pipeline behaves the same way after serialization.

In [17]:
holdout_application_inputs = (
    final_holdout_predictions_df[
        [
            "NPD_WELL_BORE_NAME",
            "DATE",
            "ridge_prediction"
        ]
    ]
    .merge(
        model_ready_df[
            [
                "NPD_WELL_BORE_NAME",
                "DATE"
            ]
            +
            CORE_NUMERIC_FEATURES
        ],

        on=[
            "NPD_WELL_BORE_NAME",
            "DATE"
        ],

        how="left",

        validate="one_to_one"
    )
)


assert len(
    holdout_application_inputs
) == 80


missing_input_values = (
    holdout_application_inputs[
        APPLICATION_MODEL_FEATURES
    ]
    .isna()
    .sum()
)


display(
    missing_input_values
    .rename(
        "missing_values"
    )
    .to_frame()
)


print(
    "Matched final-holdout origins:",
    len(
        holdout_application_inputs
    )
)


print(
    "Total missing predictor values:",
    int(
        missing_input_values.sum()
    )
)

,missing_values
NPD_WELL_BORE_NAME,0
oil_volume,0
production_age_months,0
oil_per_on_stream_hour,2
water_cut_pct,2
gas_oil_ratio,2
on_stream_hours,0
oil_volume_lag_1,0
oil_volume_lag_2,0
oil_volume_lag_3,0


Matched final-holdout origins: 80
Total missing predictor values: 6


### 5.4 Prediction Reproducibility

The saved pipeline is used to reproduce the Ridge predictions for the same 80 final-holdout observations.

The reproduced values are compared with the predictions stored by Notebook 03. This is a reproducibility check rather than a new model evaluation.

In [18]:
reproduced_ridge_predictions = (
    frozen_ridge_pipeline.predict(
        holdout_application_inputs[
            APPLICATION_MODEL_FEATURES
        ]
    )
)


stored_ridge_predictions = (
    holdout_application_inputs[
        "ridge_prediction"
    ]
    .to_numpy()
)


prediction_differences = (
    reproduced_ridge_predictions
    -
    stored_ridge_predictions
)


maximum_absolute_prediction_difference = (
    np.abs(
        prediction_differences
    )
    .max()
)


mean_absolute_prediction_difference = (
    np.abs(
        prediction_differences
    )
    .mean()
)


print(
    "Reproduced predictions:",
    len(
        reproduced_ridge_predictions
    )
)


print(
    "Maximum absolute difference:",
    maximum_absolute_prediction_difference
)


print(
    "Mean absolute difference:",
    mean_absolute_prediction_difference
)


assert np.allclose(
    reproduced_ridge_predictions,
    stored_ridge_predictions,
    rtol=1e-9,
    atol=1e-6
)


print(
    "\nPASS: serialized Ridge pipeline reproduces the frozen final-holdout predictions."
)

Reproduced predictions: 80
Maximum absolute difference: 7.275957614183426e-12
Mean absolute difference: 3.3386626796527705e-13

PASS: serialized Ridge pipeline reproduces the frozen final-holdout predictions.


### 5.5 Structural Missingness and Pipeline Compatibility

A small number of final-holdout rows contain unavailable production-intensity and fluid-ratio values.

These rows are inspected to identify their production context and to confirm that the saved preprocessing pipeline can accept the same missing-value structure that occurred during the original evaluation.

In [19]:
missing_predictor_row_mask = (
    holdout_application_inputs[
        APPLICATION_MODEL_FEATURES
    ]
    .isna()
    .any(
        axis=1
    )
)


missing_predictor_rows = (
    holdout_application_inputs.loc[
        missing_predictor_row_mask,
        [
            "NPD_WELL_BORE_NAME",
            "DATE",
            "oil_volume",
            "on_stream_hours",
            "oil_per_on_stream_hour",
            "water_cut_pct",
            "gas_oil_ratio",
            "ridge_prediction"
        ]
    ]
    .copy()
)


display(
    missing_predictor_rows
)


print(
    "Rows containing missing predictors:",
    len(
        missing_predictor_rows
    )
)


print(
    "Total missing predictor values:",
    int(
        missing_input_values.sum()
    )
)


print(
    "All reproduced predictions finite:",
    bool(
        np.isfinite(
            reproduced_ridge_predictions
        )
        .all()
    )
)


print(
    "Negative reproduced Ridge predictions:",
    int(
        (
            reproduced_ridge_predictions
            <
            0
        )
        .sum()
    )
)


assert len(
    missing_predictor_rows
) == 2


assert (
    missing_predictor_rows[
        "oil_volume"
    ]
    ==
    0
).all()


assert (
    missing_predictor_rows[
        "on_stream_hours"
    ]
    ==
    0
).all()


assert (
    missing_predictor_rows[
        [
            "oil_per_on_stream_hour",
            "water_cut_pct",
            "gas_oil_ratio"
        ]
    ]
    .isna()
    .all()
    .all()
)


assert np.isfinite(
    reproduced_ridge_predictions
).all()


assert (
    reproduced_ridge_predictions
    <
    0
).sum() == 12


print(
    "\nPASS: structural missing predictor values are "
    "preserved and accepted by the frozen pipeline, "
    "with finite reproduced predictions."
)

,NPD_WELL_BORE_NAME,DATE,oil_volume,on_stream_hours,oil_per_on_stream_hour,water_cut_pct,gas_oil_ratio,ridge_prediction
62,15/9-F-14,2016-08-01,0.0,0.0,NaN,NaN,NaN,21111.810403
79,15/9-F-15 D,2016-08-01,0.0,0.0,NaN,NaN,NaN,10798.546118


Rows containing missing predictors: 2
Total missing predictor values: 6
All reproduced predictions finite: True
Negative reproduced Ridge predictions: 12

PASS: structural missing predictor values are preserved and accepted by the frozen pipeline, with finite reproduced predictions.


## 6. Application Metadata, Methodology and Limitations Manifest

Structured project metadata is prepared for the Streamlit application.

The metadata records the frozen forecasting objective, model specification, benchmark, temporal-validation design, final-holdout results, application scope, and important analytical limitations.

These records allow the application to communicate how the forecasting system was developed and evaluated without embedding methodological assumptions directly within the user-interface code.

The metadata does not introduce new analytical conclusions. It summarizes results and methodological decisions already established in Notebooks 02, 03, 04, and the preceding sections of Notebook 05.

### 6.1 Frozen Final-Holdout Metadata Metrics

The main final-holdout performance indicators are reproduced from the frozen prediction records for inclusion in the application metadata.

These values describe the already-completed final evaluation and do not reopen model selection or tuning.

In [20]:
holdout_actual = (
    app_forecast_comparison_df[
        "target_next_month_oil_volume"
    ]
    .to_numpy()
)


ridge_holdout_prediction = (
    app_forecast_comparison_df[
        "ridge_prediction"
    ]
    .to_numpy()
)


persistence_holdout_prediction = (
    app_forecast_comparison_df[
        "persistence_prediction"
    ]
    .to_numpy()
)


def calculate_frozen_metrics(
    actual,
    prediction
):
    errors = (
        prediction
        -
        actual
    )

    absolute_errors = np.abs(
        errors
    )

    mae = (
        absolute_errors
        .mean()
    )

    rmse = np.sqrt(
        np.mean(
            errors ** 2
        )
    )

    denominator = np.sum(
        (
            actual
            -
            np.mean(
                actual
            )
        )
        ** 2
    )

    r_squared = (
        1
        -
        (
            np.sum(
                errors ** 2
            )
            /
            denominator
        )
    )

    wape = (
        np.sum(
            absolute_errors
        )
        /
        np.sum(
            np.abs(
                actual
            )
        )
        *
        100
    )

    return {
        "MAE": float(
            mae
        ),
        "RMSE": float(
            rmse
        ),
        "R2": float(
            r_squared
        ),
        "WAPE_pct": float(
            wape
        )
    }


ridge_holdout_metrics = (
    calculate_frozen_metrics(
        holdout_actual,
        ridge_holdout_prediction
    )
)


persistence_holdout_metrics = (
    calculate_frozen_metrics(
        holdout_actual,
        persistence_holdout_prediction
    )
)


frozen_holdout_metric_summary = (
    pd.DataFrame(
        [
            {
                "approach":
                    "Persistence",

                **persistence_holdout_metrics
            },
            {
                "approach":
                    "Ridge Regression",

                **ridge_holdout_metrics
            }
        ]
    )
)


display(
    frozen_holdout_metric_summary.round(
        {
            "MAE": 4,
            "RMSE": 4,
            "R2": 4,
            "WAPE_pct": 4
        }
    )
)


assert np.isclose(
    ridge_holdout_metrics[
        "MAE"
    ],
    3370.2774,
    atol=0.001
)


assert np.isclose(
    ridge_holdout_metrics[
        "RMSE"
    ],
    4577.9936,
    atol=0.001
)


assert np.isclose(
    persistence_holdout_metrics[
        "MAE"
    ],
    1786.6786,
    atol=0.001
)


assert np.isclose(
    persistence_holdout_metrics[
        "RMSE"
    ],
    2896.8847,
    atol=0.001
)


print(
    "\nPASS: application metadata metrics reproduce "
    "the frozen final-holdout results."
)

,approach,MAE,RMSE,R2,WAPE_pct
0,Persistence,1786.6786,2896.8847,0.9379,17.5492
1,Ridge Regression,3370.2774,4577.9936,0.8448,33.1037



PASS: application metadata metrics reproduce the frozen final-holdout results.


### 6.2 Application Methodology Metadata

A structured metadata record is created to describe the forecasting task, model configuration, validation framework, final evaluation, application scope, and analytical provenance.

The metadata is intended for display in application sections such as Methodology, Model Information, Data Information, and Forecast Reliability.

In [21]:
application_metadata = {
    "project": {
        "project_name":
            "Volve Oil Production Data Science Project",

        "case_study":
            "Volve Field",

        "application_type":
            "Historical production analysis, forecasting "
            "and decision-support application"
    },

    "forecasting_task": {
        "target":
            "Next-month monthly oil volume",

        "forecast_horizon":
            "One month ahead",

        "forecast_origin":
            "End of current producer-well month",

        "model_scope":
            "Pooled multi-well forecasting with "
            "producer-well identity retained"
    },

    "selected_model": {
        "algorithm":
            "Ridge Regression",

        "ridge_alpha":
            10.0,

        "numeric_feature_count":
            len(
                CORE_NUMERIC_FEATURES
            ),

        "categorical_feature_count":
            len(
                CATEGORICAL_FEATURES
            ),

        "total_input_columns":
            len(
                APPLICATION_MODEL_FEATURES
            ),

        "serialized_model":
            FINAL_MODEL_PATH.name
    },

    "benchmark": {
        "approach":
            "Naive persistence",

        "definition":
            "Current-month oil volume is used as "
            "the forecast for the following month.",

        "final_observation":
            "Persistence was the strongest observed "
            "out-of-sample approach on the frozen "
            "final holdout."
    },

    "validation": {
        "strategy":
            "Temporal validation",

        "development_design":
            "Expanding-window validation within "
            "the development period",

        "random_split_used":
            False,

        "final_holdout_origin_start":
            str(
                app_forecast_comparison_df[
                    "DATE"
                ]
                .min()
                .date()
            ),

        "final_holdout_origin_end":
            str(
                app_forecast_comparison_df[
                    "DATE"
                ]
                .max()
                .date()
            ),

        "final_holdout_observations":
            int(
                len(
                    app_forecast_comparison_df
                )
            ),

        "final_holdout_wells":
            int(
                app_forecast_comparison_df[
                    "NPD_WELL_BORE_NAME"
                ]
                .nunique()
            ),

        "post_holdout_retuning":
            False
    },

    "final_holdout_results": {
        "persistence": {
            "MAE":
                persistence_holdout_metrics[
                    "MAE"
                ],

            "RMSE":
                persistence_holdout_metrics[
                    "RMSE"
                ],

            "R2":
                persistence_holdout_metrics[
                    "R2"
                ],

            "WAPE_pct":
                persistence_holdout_metrics[
                    "WAPE_pct"
                ]
        },

        "ridge_regression": {
            "MAE":
                ridge_holdout_metrics[
                    "MAE"
                ],

            "RMSE":
                ridge_holdout_metrics[
                    "RMSE"
                ],

            "R2":
                ridge_holdout_metrics[
                    "R2"
                ],

            "WAPE_pct":
                ridge_holdout_metrics[
                    "WAPE_pct"
                ],

            "negative_predictions":
                int(
                    (
                        ridge_holdout_prediction
                        <
                        0
                    )
                    .sum()
                )
        }
    },

    "application_scope": {
        "core_producer_wells":
            sorted(
                EXPECTED_CORE_WELLS
            ),

        "historical_application_rows":
            int(
                len(
                    application_history_df
                )
            ),

        "forecast_comparison_rows":
            int(
                len(
                    app_forecast_comparison_df
                )
            ),

        "monitoring_summary_rows":
            int(
                len(
                    app_well_monitoring_df
                )
            ),

        "latest_monitoring_origin":
            "Latest available forecasting origin "
            "is determined separately for each well."
    },

    "provenance": {
        "notebook_02":
            "Production decline analysis, forecasting "
            "features and modelling-ready dataset",

        "notebook_03":
            "Predictive model development, temporal "
            "validation and frozen final holdout",

        "notebook_04":
            "Operational production interpretation "
            "and dashboard-ready monitoring outputs",

        "notebook_05":
            "Application integration, artifact "
            "validation and Streamlit readiness"
    }
}


print(
    json.dumps(
        application_metadata,
        indent=4
    )
)

{
    "project": {
        "project_name": "Volve Oil Production Data Science Project",
        "case_study": "Volve Field",
        "application_type": "Historical production analysis, forecasting and decision-support application"
    },
    "forecasting_task": {
        "target": "Next-month monthly oil volume",
        "forecast_horizon": "One month ahead",
        "forecast_origin": "End of current producer-well month",
        "model_scope": "Pooled multi-well forecasting with producer-well identity retained"
    },
    "selected_model": {
        "algorithm": "Ridge Regression",
        "ridge_alpha": 10.0,
        "numeric_feature_count": 10,
        "categorical_feature_count": 1,
        "total_input_columns": 11,
        "serialized_model": "ridge_next_month_oil_forecast_pipeline.joblib"
    },
    "benchmark": {
        "approach": "Naive persistence",
        "definition": "Current-month oil volume is used as the forecast for the following month.",
        "final_observatio

### 6.3 Application Limitations Register

The principal analytical and application limitations are recorded explicitly for use in the Streamlit interface and final project documentation.

The limitations distinguish forecasting evidence from engineering interpretation and identify conditions under which the model outputs should be treated cautiously.

In [22]:
application_limitations_df = pd.DataFrame(
    [
        {
            "limitation_id": "L01",
            "category": "Dataset",
            "limitation":
                "The monthly forecasting dataset contains "
                "a relatively limited number of valid "
                "producer-well forecasting observations.",
            "application_implication":
                "Performance estimates should be interpreted "
                "within the historical Volve case-study scope."
        },

        {
            "limitation_id": "L02",
            "category": "Generalization",
            "limitation":
                "The selected Ridge Regression model did not "
                "outperform the persistence benchmark on the "
                "untouched final holdout.",
            "application_implication":
                "The application presents both approaches and "
                "does not describe Ridge Regression as superior."
        },

        {
            "limitation_id": "L03",
            "category": "Model behaviour",
            "limitation":
                "Ridge Regression produced 12 negative forecasts "
                "during the 80-observation final holdout.",
            "application_implication":
                "Negative predictions are preserved as model "
                "diagnostics and are not retrospectively clipped."
        },

        {
            "limitation_id": "L04",
            "category": "Production state",
            "limitation":
                "Zero-production and zero-on-stream observations "
                "can contain structurally unavailable production "
                "intensity and fluid-ratio indicators.",
            "application_implication":
                "These values remain missing rather than being "
                "manually replaced with artificial values."
        },

        {
            "limitation_id": "L05",
            "category": "Zero-production forecasting",
            "limitation":
                "The frozen Ridge pipeline can produce finite but "
                "large positive forecasts when the current well "
                "state contains zero production.",
            "application_implication":
                "Computational compatibility does not imply "
                "forecast reliability under zero-production "
                "conditions."
        },

        {
            "limitation_id": "L06",
            "category": "Operational interpretation",
            "limitation":
                "Observed production trends, water cut, gas-oil "
                "ratio and operating-time changes do not establish "
                "engineering causation.",
            "application_implication":
                "The application provides descriptive decision "
                "support rather than automated engineering advice."
        },

        {
            "limitation_id": "L07",
            "category": "Application scope",
            "limitation":
                "The application represents historical Volve data "
                "and is not connected to a live production system.",
            "application_implication":
                "Displayed monitoring states are historical "
                "case-study observations rather than real-time "
                "field conditions."
        },

        {
            "limitation_id": "L08",
            "category": "Validation",
            "limitation":
                "Random train-test splitting was intentionally "
                "avoided because the forecasting problem is "
                "temporally ordered.",
            "application_implication":
                "Reported results reflect temporal validation and "
                "should not be compared directly with results "
                "produced using shuffled evaluation designs."
        }
    ]
)


assert (
    application_limitations_df[
        "limitation_id"
    ]
    .is_unique
)


assert len(
    application_limitations_df
) == 8


display(
    application_limitations_df
)

,limitation_id,category,limitation,application_implication
0,L01,Dataset,The monthly forecasting dataset contains a rel...,Performance estimates should be interpreted wi...
1,L02,Generalization,The selected Ridge Regression model did not ou...,The application presents both approaches and d...
2,L03,Model behaviour,Ridge Regression produced 12 negative forecast...,Negative predictions are preserved as model di...
3,L04,Production state,Zero-production and zero-on-stream observation...,These values remain missing rather than being ...
4,L05,Zero-production forecasting,The frozen Ridge pipeline can produce finite b...,Computational compatibility does not imply for...
5,L06,Operational interpretation,"Observed production trends, water cut, gas-oil...",The application provides descriptive decision ...
6,L07,Application scope,The application represents historical Volve da...,Displayed monitoring states are historical cas...
7,L08,Validation,Random train-test splitting was intentionally ...,Reported results reflect temporal validation a...


### 6.4 Export of Application Metadata and Limitations

The validated methodology metadata and limitations register are exported as application artifacts.

The JSON metadata file provides machine-readable project and model information, while the limitations CSV provides structured explanatory content for the Streamlit interface and final documentation.

In [23]:
APPLICATION_METADATA_PATH = (
    NOTEBOOK05_OUTPUT_DIR
    /
    "app_metadata.json"
)


APPLICATION_LIMITATIONS_PATH = (
    NOTEBOOK05_OUTPUT_DIR
    /
    "app_limitations.csv"
)


with open(
    APPLICATION_METADATA_PATH,
    "w",
    encoding="utf-8"
) as metadata_file:
    json.dump(
        application_metadata,
        metadata_file,
        indent=4
    )


application_limitations_df.to_csv(
    APPLICATION_LIMITATIONS_PATH,
    index=False
)


assert (
    APPLICATION_METADATA_PATH.exists()
)


assert (
    APPLICATION_LIMITATIONS_PATH.exists()
)


with open(
    APPLICATION_METADATA_PATH,
    "r",
    encoding="utf-8"
) as metadata_file:
    exported_metadata_check = (
        json.load(
            metadata_file
        )
    )


exported_limitations_check = (
    pd.read_csv(
        APPLICATION_LIMITATIONS_PATH
    )
)


assert (
    exported_metadata_check[
        "selected_model"
    ][
        "algorithm"
    ]
    ==
    "Ridge Regression"
)


assert (
    exported_metadata_check[
        "validation"
    ][
        "final_holdout_observations"
    ]
    ==
    80
)


assert (
    exported_metadata_check[
        "final_holdout_results"
    ][
        "ridge_regression"
    ][
        "negative_predictions"
    ]
    ==
    12
)


assert len(
    exported_limitations_check
) == 8


print(
    "Saved:",
    APPLICATION_METADATA_PATH
)


print(
    "Saved:",
    APPLICATION_LIMITATIONS_PATH
)


print(
    "\nMetadata sections:",
    len(
        exported_metadata_check
    )
)


print(
    "Registered limitations:",
    len(
        exported_limitations_check
    )
)


print(
    "\nPASS: application metadata and limitations "
    "artifacts exported successfully."
)

Saved: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration\app_metadata.json
Saved: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration\app_limitations.csv

Metadata sections: 8
Registered limitations: 8

PASS: application metadata and limitations artifacts exported successfully.


## 7. Final Application Integration Check

Before closing the notebook, the application datasets, metadata files, limitations register, and saved model are checked together.

The purpose of this section is to confirm that the exported files are present, have the expected structure, and are ready to be consumed by the Streamlit application.

### 7.1 Final Application Artifact Inventory

The application artifacts are assembled into a single inventory describing their locations, roles, and expected structures.

The serialized forecasting pipeline is included as an application dependency but remains the frozen model artifact originally produced in Notebook 03.

In [24]:
final_application_artifacts = [
    {
        "artifact":
            "Historical production data",

        "path":
            APPLICATION_HISTORY_PATH,

        "expected_rows":
            295,

        "role":
            "Historical producer-well visualization "
            "and production context"
    },

    {
        "artifact":
            "Forecast comparison data",

        "path":
            APPLICATION_FORECAST_PATH,

        "expected_rows":
            80,

        "role":
            "Frozen final-holdout comparison of observed, "
            "persistence and Ridge forecasts"
    },

    {
        "artifact":
            "Well monitoring data",

        "path":
            APPLICATION_MONITORING_PATH,

        "expected_rows":
            5,

        "role":
            "Latest producer-well production condition "
            "and forecast-reliability context"
    },

    {
        "artifact":
            "Application metadata",

        "path":
            APPLICATION_METADATA_PATH,

        "expected_rows":
            None,

        "role":
            "Forecasting methodology, model specification, "
            "validation design and analytical provenance"
    },

    {
        "artifact":
            "Application limitations",

        "path":
            APPLICATION_LIMITATIONS_PATH,

        "expected_rows":
            8,

        "role":
            "Structured model, dataset and application "
            "limitations"
    },

    {
        "artifact":
            "Frozen Ridge pipeline",

        "path":
            FINAL_MODEL_PATH,

        "expected_rows":
            None,

        "role":
            "Serialized selected machine-learning pipeline "
            "for application integration"
    }
]


final_artifact_inventory = pd.DataFrame(
    [
        {
            "artifact":
                item[
                    "artifact"
                ],

            "exists":
                item[
                    "path"
                ]
                .exists(),

            "expected_rows":
                item[
                    "expected_rows"
                ],

            "role":
                item[
                    "role"
                ],

            "path":
                str(
                    item[
                        "path"
                    ]
                )
        }
        for item
        in final_application_artifacts
    ]
)


display(
    final_artifact_inventory
)


assert (
    final_artifact_inventory[
        "exists"
    ]
    .all()
)


print(
    "Verified application dependencies:",
    int(
        final_artifact_inventory[
            "exists"
        ]
        .sum()
    ),
    "/",
    len(
        final_artifact_inventory
    )
)


print(
    "\nPASS: all final application dependencies are available."
)

,artifact,exists,expected_rows,role,path
0,Historical production data,True,295.0,Historical producer-well visualization and pro...,F:\DataScience_Projects\Volve_Oil_Production\o...
1,Forecast comparison data,True,80.0,"Frozen final-holdout comparison of observed, p...",F:\DataScience_Projects\Volve_Oil_Production\o...
2,Well monitoring data,True,5.0,Latest producer-well production condition and ...,F:\DataScience_Projects\Volve_Oil_Production\o...
3,Application metadata,True,NaN,"Forecasting methodology, model specification, ...",F:\DataScience_Projects\Volve_Oil_Production\o...
4,Application limitations,True,8.0,"Structured model, dataset and application limi...",F:\DataScience_Projects\Volve_Oil_Production\o...
5,Frozen Ridge pipeline,True,NaN,Serialized selected machine-learning pipeline ...,F:\DataScience_Projects\Volve_Oil_Production\m...


Verified application dependencies: 6 / 6

PASS: all final application dependencies are available.


### 7.2 Final Application Data-Contract Verification

The exported application datasets are reloaded from disk and checked against their expected row counts, producer-well coverage, forecast structure, monitoring structure, and metadata definitions.

This verifies the exported files themselves rather than relying only on the in-memory DataFrames used to create them.

In [25]:
final_history_check = pd.read_csv(
    APPLICATION_HISTORY_PATH,
    parse_dates=[
        "DATE"
    ]
)


final_forecast_check = pd.read_csv(
    APPLICATION_FORECAST_PATH,
    parse_dates=[
        "DATE",
        "target_month"
    ]
)


final_monitoring_check = pd.read_csv(
    APPLICATION_MONITORING_PATH,
    parse_dates=[
        "DATE"
    ]
)


final_limitations_check = pd.read_csv(
    APPLICATION_LIMITATIONS_PATH
)


with open(
    APPLICATION_METADATA_PATH,
    "r",
    encoding="utf-8"
) as metadata_file:
    final_metadata_check = json.load(
        metadata_file
    )


assert len(
    final_history_check
) == 295


assert len(
    final_forecast_check
) == 80


assert len(
    final_monitoring_check
) == 5


assert len(
    final_limitations_check
) == 8


assert (
    set(
        final_history_check[
            "NPD_WELL_BORE_NAME"
        ]
        .unique()
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    set(
        final_forecast_check[
            "NPD_WELL_BORE_NAME"
        ]
        .unique()
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    set(
        final_monitoring_check[
            "NPD_WELL_BORE_NAME"
        ]
        .unique()
    )
    ==
    EXPECTED_CORE_WELLS
)


assert (
    final_forecast_check[
        "ridge_negative_prediction"
    ]
    .sum()
    ==
    12
)


assert (
    final_monitoring_check[
        "zero_production_flag"
    ]
    .sum()
    ==
    2
)


assert (
    final_metadata_check[
        "validation"
    ][
        "final_holdout_observations"
    ]
    ==
    80
)


assert (
    final_metadata_check[
        "validation"
    ][
        "post_holdout_retuning"
    ]
    is False
)


assert (
    final_metadata_check[
        "benchmark"
    ][
        "approach"
    ]
    ==
    "Naive persistence"
)


assert FINAL_MODEL_PATH.stat().st_size > 0


print(
    "Historical production rows:",
    len(
        final_history_check
    )
)


print(
    "Forecast comparison rows:",
    len(
        final_forecast_check
    )
)


print(
    "Well monitoring rows:",
    len(
        final_monitoring_check
    )
)


print(
    "Registered limitations:",
    len(
        final_limitations_check
    )
)


print(
    "Negative Ridge forecasts:",
    int(
        final_forecast_check[
            "ridge_negative_prediction"
        ]
        .sum()
    )
)


print(
    "Zero-production monitoring states:",
    int(
        final_monitoring_check[
            "zero_production_flag"
        ]
        .sum()
    )
)


print(
    "\nPASS: final application data contracts are consistent."
)

Historical production rows: 295
Forecast comparison rows: 80
Well monitoring rows: 5
Registered limitations: 8
Negative Ridge forecasts: 12
Zero-production monitoring states: 2

PASS: final application data contracts are consistent.


### 7.3 Final Implementation Manifest

A final implementation manifest is created to document the analytical state handed to the Streamlit application.

The manifest records the frozen forecasting configuration, application artifacts, model dependency, final-holdout conclusion, and implementation boundaries.

It serves as a compact reproducibility record and does not modify any analytical result.

In [26]:
implementation_manifest = {
    "project":
        "Volve Oil Production Data Science Project",

    "implementation_stage":
        "Application integration complete",

    "forecasting_configuration": {
        "target":
            "Next-month monthly oil volume",

        "forecast_horizon":
            "One month ahead",

        "selected_machine_learning_model":
            "Ridge Regression",

        "ridge_alpha":
            10.0,

        "benchmark":
            "Naive persistence",

        "validation":
            "Temporal expanding-window development "
            "validation followed by one untouched "
            "final holdout",

        "post_holdout_retuning":
            False
    },

    "frozen_final_holdout": {
        "observations":
            80,

        "core_wells":
            5,

        "persistence_MAE":
            persistence_holdout_metrics[
                "MAE"
            ],

        "persistence_RMSE":
            persistence_holdout_metrics[
                "RMSE"
            ],

        "persistence_WAPE_pct":
            persistence_holdout_metrics[
                "WAPE_pct"
            ],

        "ridge_MAE":
            ridge_holdout_metrics[
                "MAE"
            ],

        "ridge_RMSE":
            ridge_holdout_metrics[
                "RMSE"
            ],

        "ridge_WAPE_pct":
            ridge_holdout_metrics[
                "WAPE_pct"
            ],

        "ridge_negative_predictions":
            12,

        "observed_out_of_sample_conclusion":
            "Persistence achieved stronger final-holdout "
            "performance than the selected Ridge Regression "
            "model."
    },

    "application_artifacts": {
        "historical_production": {
            "file":
                APPLICATION_HISTORY_PATH.name,

            "rows":
                295
        },

        "forecast_comparison": {
            "file":
                APPLICATION_FORECAST_PATH.name,

            "rows":
                80
        },

        "well_monitoring": {
            "file":
                APPLICATION_MONITORING_PATH.name,

            "rows":
                5
        },

        "metadata": {
            "file":
                APPLICATION_METADATA_PATH.name
        },

        "limitations": {
            "file":
                APPLICATION_LIMITATIONS_PATH.name,

            "rows":
                8
        },

        "serialized_model": {
            "file":
                FINAL_MODEL_PATH.name,

            "source":
                "Notebook 03"
        }
    },

    "analytical_boundaries": {
        "new_model_training_in_notebook_05":
            False,

        "new_hyperparameter_tuning_in_notebook_05":
            False,

        "new_feature_selection_in_notebook_05":
            False,

        "final_holdout_reopened_for_model_selection":
            False,

        "negative_predictions_clipped":
            False,

        "engineering_alarm_thresholds_created":
            False,

        "causal_engineering_claims_created":
            False
    },

    "next_stage":
        "Streamlit application development"
}


IMPLEMENTATION_MANIFEST_PATH = (
    NOTEBOOK05_OUTPUT_DIR
    /
    "application_implementation_manifest.json"
)


with open(
    IMPLEMENTATION_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as manifest_file:
    json.dump(
        implementation_manifest,
        manifest_file,
        indent=4
    )


assert (
    IMPLEMENTATION_MANIFEST_PATH.exists()
)


with open(
    IMPLEMENTATION_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as manifest_file:
    implementation_manifest_check = (
        json.load(
            manifest_file
        )
    )


assert (
    implementation_manifest_check[
        "forecasting_configuration"
    ][
        "post_holdout_retuning"
    ]
    is False
)


assert (
    implementation_manifest_check[
        "analytical_boundaries"
    ][
        "final_holdout_reopened_for_model_selection"
    ]
    is False
)


assert (
    implementation_manifest_check[
        "next_stage"
    ]
    ==
    "Streamlit application development"
)


print(
    "Saved:",
    IMPLEMENTATION_MANIFEST_PATH
)


print(
    "\nPASS: final implementation manifest exported successfully."
)

Saved: F:\DataScience_Projects\Volve_Oil_Production\outputs\05_application_integration\application_implementation_manifest.json

PASS: final implementation manifest exported successfully.


### 7.4 Final Notebook 05 Artifact Verification

All artifacts produced specifically during the application-integration workflow are verified one final time before Notebook 05 is closed.

In [27]:
required_notebook05_artifacts = [
    APPLICATION_HISTORY_PATH,
    APPLICATION_FORECAST_PATH,
    APPLICATION_MONITORING_PATH,
    APPLICATION_METADATA_PATH,
    APPLICATION_LIMITATIONS_PATH,
    IMPLEMENTATION_MANIFEST_PATH
]


notebook05_artifact_verification = pd.DataFrame(
    [
        {
            "artifact":
                path.name,

            "exists":
                path.exists(),

            "size_bytes":
                (
                    path.stat().st_size
                    if path.exists()
                    else 0
                )
        }
        for path
        in required_notebook05_artifacts
    ]
)


display(
    notebook05_artifact_verification
)


assert (
    notebook05_artifact_verification[
        "exists"
    ]
    .all()
)


assert (
    notebook05_artifact_verification[
        "size_bytes"
    ]
    >
    0
).all()


verified_notebook05_artifacts = int(
    notebook05_artifact_verification[
        "exists"
    ]
    .sum()
)


print(
    "Verified Notebook 05 artifacts:",
    verified_notebook05_artifacts,
    "/",
    len(
        required_notebook05_artifacts
    )
)


print(
    "\nPASS: all required Notebook 05 "
    "application-integration artifacts exist."
)

,artifact,exists,size_bytes
0,app_historical_production.csv,True,45198
1,app_forecast_comparison.csv,True,11062
2,app_well_monitoring.csv,True,2037
3,app_metadata.json,True,2920
4,app_limitations.csv,True,1954
5,application_implementation_manifest.json,True,2335


Verified Notebook 05 artifacts: 6 / 6

PASS: all required Notebook 05 application-integration artifacts exist.


## 8. Notebook Conclusion

Notebook 05 completed the handoff from the analytical workflow to the application layer.

The final application inputs consist of 295 historical producer-well observations, 80 final-holdout forecast records, five well-level monitoring summaries, project metadata, a structured limitations register, and the saved Ridge Regression pipeline.

The saved pipeline was loaded successfully and reproduced the original final-holdout predictions to numerical precision. The two observations containing missing production-intensity and fluid-ratio values were zero-production, zero-on-stream cases, so those values were kept as structurally unavailable rather than manually replaced.

The forecasting conclusion remains the same as in Notebook 03. Ridge Regression was selected during development, but persistence performed better on the untouched final holdout. Both approaches are therefore retained in the application so that model performance can be presented transparently.

Notebook 05 did not retrain or retune the model, change the selected features, clip negative forecasts, or introduce engineering thresholds.

The analytical notebook workflow is now complete. The next stage is the Streamlit application.